In [1]:
library(broom)  # for tidy model output
library(ggplot2)


In [2]:
meta_df <- read.csv('../../data/metadata//pan_metadata_v5.csv')

In [15]:
head(meta_df,1)

,sample,sample.display,age,organ,dataset,hrd.origin,genotype,purity,type,grade,⋯,RS5,del.mh.prop,HRDetect,HRDetect.bootstrap.score.percentile.5,HRDetect.bootstrap.score.percentile.95.,HRDetect.isHRD,isWGD,q1,q5,timerQC_fail
,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>,<dbl>,<dbl>,<lgl>
1,0009b464-b376-4fbc-8a56-da538269a02f,SA505245,54,Ovary,PCAWG,nonHRD,NA,0.885,NA,,⋯,38.17195,0.2421053,0.02487369,0.009768601,0.05344698,FALSE,TRUE,NA,NA,FALSE


In [4]:
nrow(meta_df)

[1] 390

In [5]:
hrdtimer_df <- read.csv('../../data/output/PCAWG_SCANB_INFORM_SBS1_Age_plot.csv')

In [16]:
head(hrdtimer_df,1)

,X,sample,age,isWGD,HRDetect.isHRD,organ,G,SBS1_Early,SBS1_Late,SBS1_NA,SBS1_total,Cohort,scaled_SBS1
,<int>,<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>
1,0,01658141-8398-4585-9f0f-8355dd9b0604,62,True,False,Breast,2.421293,311.4406,376.4261,306.655,994.5216,PCAWG,0.1369133


In [17]:
meta_breast_df <- subset(meta_df, organ=='Breast')
head(meta_breast_df,1)

,sample,sample.display,age,organ,dataset,hrd.origin,genotype,purity,type,grade,⋯,RS5,del.mh.prop,HRDetect,HRDetect.bootstrap.score.percentile.5,HRDetect.bootstrap.score.percentile.95.,HRDetect.isHRD,isWGD,q1,q5,timerQC_fail
,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>,<dbl>,<dbl>,<lgl>
2,01658141-8398-4585-9f0f-8355dd9b0604,SA17443,62,Breast,PCAWG,nonHRD,NA,0.539,ER+,,⋯,33.53927,0.2092199,0.006929235,0.002685464,0.014493,FALSE,TRUE,NA,NA,FALSE


In [9]:
meta_breast_df_m <- merge(hrdtimer_df, meta_breast_df, by='sample')
meta_breast_df_m$Age.decade <- meta_breast_df_m$age.x/10

In [18]:
head(meta_breast_df_m,1)

,sample,X,age.x,isWGD.x,HRDetect.isHRD.x,organ.x,G,SBS1_Early,SBS1_Late,SBS1_NA,⋯,del.mh.prop,HRDetect,HRDetect.bootstrap.score.percentile.5,HRDetect.bootstrap.score.percentile.95.,HRDetect.isHRD.y,isWGD.y,q1,q5,timerQC_fail,Age.decade
,<chr>,<int>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>,<dbl>,<dbl>,<lgl>,<dbl>
1,01658141-8398-4585-9f0f-8355dd9b0604,0,62,True,False,Breast,2.421293,311.4406,376.4261,306.655,⋯,0.2092199,0.006929235,0.002685464,0.014493,FALSE,TRUE,NA,NA,FALSE,6.2


In [11]:
meta_breast_df_m_g23 <- subset(meta_breast_df_m, grade %in% c('G2','G3'))

model <- lm(scaled_SBS1 ~  Age.decade	+  grade + type + dataset, data = meta_breast_df_m_g23)
# Obtain the summary of the model
summary_model <- summary(model)

In [12]:
summary_model


Call:
lm(formula = scaled_SBS1 ~ Age.decade + grade + type + dataset, 
    data = meta_breast_df_m_g23)

Residuals:
      Min        1Q    Median        3Q       Max 
-0.049188 -0.012045 -0.001672  0.008663  0.067810 

Coefficients:
               Estimate Std. Error t value Pr(>|t|)    
(Intercept)  -0.0126919  0.0121718  -1.043 0.299717    
Age.decade    0.0073695  0.0015763   4.675 9.69e-06 ***
gradeG3       0.0168171  0.0069209   2.430 0.016981 *  
typeTN        0.0201615  0.0056727   3.554 0.000593 ***
datasetPCAWG  0.0107813  0.0080756   1.335 0.185052    
datasetSCANB -0.0006496  0.0078039  -0.083 0.933833    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.01946 on 95 degrees of freedom
  (1 observation deleted due to missingness)
Multiple R-squared:  0.3207,	Adjusted R-squared:  0.2849 
F-statistic:  8.97 on 5 and 95 DF,  p-value: 5.232e-07


In [13]:
coef_df <- tidy(summary_model, conf.int = TRUE)
pdf("../../data/output/output_plots/Fig3B_SBS1_burden_multivariate.pdf", width = 7, height = 7)
options(repr.plot.width=6, repr.plot.height=6)

ggplot(coef_df, aes(x = term, y = estimate)) +
  geom_point() +
  geom_errorbar(aes(ymin = conf.low, ymax = conf.high), width = 0.2) +
  labs(title = "", x = "", y = "Coefficient Estimate") +
  theme_minimal() +
  geom_hline(yintercept = 0, linetype = "dashed", color = "red") +
  #coord_flip() +  # Flip coordinates for better readability
  theme(
    plot.title = element_text(size = 20, face = "bold"),      # Title font size
    axis.title.x = element_text(size = 16),                  # X-axis title font size
    axis.title.y = element_text(size = 16),                  # Y-axis title font size
    axis.text.x = element_text(size = 14, angle = 45, hjust = 1),  # Rotate x-axis labels
    axis.text.y = element_text(size = 14),                   # Y-axis text font size
    panel.border = element_rect(color = "black", fill = NA, linewidth = 1),  # Black frame
    panel.grid = element_blank()  # Remove all grid lines
  ) +
  theme(panel.background = element_rect(fill = "white"))  # Ensures white background

dev.off()


agg_record_1975000822 
                    2

In [14]:
coef_df$estimate_diploid <- coef_df$estimate * 6000
coef_df

term,estimate,std.error,statistic,p.value,conf.low,conf.high,estimate_diploid
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
(Intercept),-0.0126919267,0.012171759,-1.04273563,2.997170e-01,-0.036855924,0.01147207,-76.151560
Age.decade,0.0073695484,0.001576288,4.67525516,9.691025e-06,0.004240221,0.01049888,44.217290
gradeG3,0.0168171213,0.006920904,2.42990235,1.698147e-02,0.003077389,0.03055685,100.902728
typeTN,0.0201614910,0.005672723,3.55411178,5.929295e-04,0.008899712,0.03142327,120.968946
datasetPCAWG,0.0107812965,0.008075609,1.33504433,1.850523e-01,-0.005250815,0.02681341,64.687779
datasetSCANB,-0.0006496276,0.007803888,-0.08324409,9.338326e-01,-0.016142303,0.01484305,-3.897765
